In [ ]:
!pip install gradio -q

In [1]:
import re
import gradio as gr
import matplotlib.pyplot as plt
import numpy as np

# =====================================================
# TOOL
# =====================================================

def compound_interest(principal, rate, years):
    return principal * ((1 + rate / 100) ** years)

# =====================================================
# SMART PARSER (ROBUST)
# =====================================================

def parse_input(text):

    text = text.lower()
    nums = list(map(float, re.findall(r'\d+\.?\d*', text)))

    principal = None
    rate = None
    years = None

    # detect %
    if "%" in text:
        rate_match = re.findall(r'(\d+\.?\d*)\s*%', text)
        if rate_match:
            rate = float(rate_match[0])

    remaining = nums.copy()
    if rate in remaining:
        remaining.remove(rate)

    if len(remaining) == 2:
        principal = max(remaining)
        years = min(remaining)

    elif len(remaining) >= 3:
        principal = max(remaining)
        years = sorted(remaining)[1]

    elif len(remaining) == 1:
        principal = remaining[0]

    return principal, rate, years

# =====================================================
# AGENT
# =====================================================

def financial_agent(message):

    principal, rate, years = parse_input(message)

    if principal is None or years is None:
        return "❌ Please enter principal and years", None

    if rate is None:
        rate = 10.0

    amount = compound_interest(principal, rate, years)

    # ---------------- GRAPH ----------------
    x = np.arange(1, int(years) + 1)
    y = [compound_interest(principal, rate, i) for i in x]

    plt.figure(figsize=(4.5, 2.5))
    plt.plot(x, y, marker='o', color='#1F77B4', linewidth=2)
    plt.title("Investment Growth Over Time")
    plt.xlabel("Years")
    plt.ylabel("Value (₹)")
    plt.grid(True, alpha=0.3)

    result = f"""
💰 Principal: ₹{principal:,.0f}
📈 Rate: {rate}%
⏳ Years: {years}

💎 Final Value: ₹{amount:,.2f}
"""

    return result, plt

# =====================================================
# 🎨 ATTRACTIVE UI (CELL 2)
# =====================================================

custom_css = """
#title {
    text-align: center;
    font-size: 30px;
    font-weight: bold;
    color: #1F4E79;
    margin-bottom: 10px;
}

.card {
    border-radius: 14px;
    padding: 15px;
    background: #f7fbff;
    border: 1px solid #d6e6ff;
    margin-bottom: 10px;
}

#submit_btn {
    background: linear-gradient(90deg, #1F77B4, #2ECC71);
    color: white;
    font-size: 16px;
    font-weight: bold;
    border-radius: 12px;
    padding: 10px;
}
"""

with gr.Blocks(css=custom_css) as demo:

    gr.Markdown("<div id='title'>🤖 Financial Advisor Agent</div>")

    # =========================
    # QUESTION CARD
    # =========================
    with gr.Group(elem_classes=["card"]):
        gr.Markdown("### 🟦 Your Question")
        msg = gr.Textbox(
            placeholder="I have 200000 for 5 years at 12% interest",
            lines=1
        )

    # =========================
    # SUBMIT BUTTON (HIGHLIGHTED)
    # =========================
    submit = gr.Button("🚀 CALCULATE", elem_id="submit_btn")

    # =========================
    # RESPONSE CARD
    # =========================
    with gr.Group(elem_classes=["card"]):
        gr.Markdown("### 🟩 Agent Response")
        answer = gr.Textbox(lines=5)

    # =========================
    # GRAPH CARD
    # =========================
    with gr.Group(elem_classes=["card"]):
        gr.Markdown("### 📊 Growth Chart")
        graph = gr.Plot()

    # =========================
    # RUN FUNCTION
    # =========================
    def run(message):
        return financial_agent(message)

    submit.click(
        run,
        inputs=msg,
        outputs=[answer, graph]
    )

demo.launch(share=True, debug=False)

/tmp/ipykernel_55835/2230985462.py:117: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://542e3fd8ca58dc00f3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
